# Sprint 6 — Handoff breve del Rol API Engineer

En este Sprint se implementó un servicio **FastAPI dockerizado** para exponer el modelo final de predicción de cancelaciones hoteleras. La API cuenta con endpoints técnicos para health check, versión y predicción, además de una interfaz web tipo formulario para usuarios no técnicos. También se incorporaron validaciones de entrada, manejo de errores con códigos HTTP claros, Dockerfile con `python:3.10-slim`, `requirements.txt` ajustado y `.dockerignore` para optimizar el build.

---

## Cumplimiento de requisitos del rol

| Requisito del Rol API Engineer | Estado | Evidencia / implementación |
|---|---:|---|
| 1. Crear el servicio FastAPI con endpoints `/health`, `/predict` y `/version`. | ✅ Cumplido | Implementado en `api/main.py`. Se probaron los endpoints y también se agregó `/` como interfaz amigable tipo formulario. |
| 2. Definir los schemas de entrada/salida con Pydantic según `handoff/contracts/`. | ✅ Cumplido | Implementado en `api/schemas.py` mediante `RawPredictionRequest`, `PredictionResponse`, `HealthResponse` y `VersionResponse`. Se validan tipos, rangos y categorías permitidas. |
| 3. Cargar el modelo y el preproc pipeline `.pkl` una sola vez al iniciar el proceso. | ✅ Cumplido | `api/inference.py` carga `models/final_pipeline.pkl` una sola vez con `joblib.load`. Este pipeline contiene `preprocessor` + `clf`. |
| 4. Implementar validación de inputs y manejo de errores con códigos HTTP claros. | ✅ Cumplido | Pydantic devuelve `422` para inputs inválidos; reglas de negocio pueden devolver `400`; errores internos devuelven `500`. Los mensajes se muestran en español con campo, mensaje y ayuda específica. |
| 5. Escribir el Dockerfile con imagen `python:3.10-slim` y `requirements.txt` ajustado. | ✅ Cumplido | Se crearon `api/Dockerfile` y `api/requirements.txt`. La imagen usa `python:3.10-slim` y contiene las dependencias necesarias para ejecutar la API. |
| 6. Probar localmente con `docker build` + `docker run` y `curl` contra los contratos. | ✅ Cumplido | Se construyó la imagen `hotel-cancel-api:latest`, se ejecutó el contenedor y `curl http://localhost:8001/health` devolvió HTTP `200 OK` con `{"status":"ok"}`. También se probó la interfaz en navegador. |

---

## Cumplimiento de buenas prácticas recomendadas

| Buena práctica recomendada | Estado | Evidencia / implementación |
|---|---:|---|
| 1. Usar `python:3.10-slim`. | ✅ Cumplido | Definido en `api/Dockerfile` con `FROM python:3.10-slim`. |
| 2. Instalar primero `requirements.txt` y luego copiar el código para aprovechar caché de Docker. | ✅ Cumplido | El Dockerfile copia primero `api/requirements.txt`, instala dependencias y luego copia `api/`. En rebuilds se observaron pasos en caché. |
| 3. Correr como usuario no-root (`USER appuser`). | ✅ Cumplido | El Dockerfile crea el usuario `appuser`, cambia ownership de `/app` y ejecuta el contenedor con `USER appuser`. |
| 4. Exponer un único puerto (`EXPOSE 8000`). | ✅ Cumplido | El Dockerfile expone solo el puerto interno `8000`. En local se puede mapear a `8001:8000` sin cambiar el puerto interno del contenedor. |
| 5. Definir `HEALTHCHECK` para que ECS/ALB sepa si el contenedor está sano. | ✅ Cumplido | El Dockerfile incluye un `HEALTHCHECK` que consulta `http://localhost:8000/health`. |
| 6. No incluir modelos innecesarios en la imagen si pesan mucho; montar desde S3 o volumen si aplica. | ✅ Cumplido para MVP | La imagen copia únicamente `models/final_pipeline.pkl`. Se excluyen otros modelos con `.dockerignore`. Para producción/cloud se recomienda evaluar S3 versionado si el modelo crece o se desea desacoplar artefactos. |

---

## Evidencias principales

| Evidencia | Resultado |
|---|---|
| Validación del pipeline final | `final_pipeline.pkl` contiene `dict_keys(['preprocessor', 'clf'])`. |
| Build Docker | `docker build -f api/Dockerfile -t hotel-cancel-api .` creó la imagen `hotel-cancel-api:latest`. |
| `.dockerignore` funcionando | El build mostró `load .dockerignore` y un contexto reducido de aproximadamente `577B`. |
| Imagen Docker creada | `docker images` mostró `hotel-cancel-api:latest`. |
| Contenedor ejecutado | Se ejecutó con `docker run --rm -p 8001:8000 hotel-cancel-api`. |
| Health check dentro del contenedor | `curl http://localhost:8001/health` devolvió `StatusCode: 200` y `{"status":"ok"}`. |
| Interfaz amigable | `http://localhost:8001/` permitió generar predicciones desde un formulario. |
| Validación de errores | Inputs inválidos devuelven errores `422` con mensajes claros en español. |

---

## Conclusión breve

El rol de **API Engineer** queda cumplido. Se entregó una API FastAPI funcional, validada, dockerizada, con endpoints técnicos requeridos, interfaz amigable tipo formulario, manejo de errores claro, carga eficiente del pipeline final y estructura lista para que los siguientes roles continúen con despliegue cloud, integración del dashboard y monitoreo.

# Sprint 6 — Handoff breve del Rol API Engineer

En este Sprint se implementó una **API FastAPI dockerizada** para usar el modelo final de predicción de cancelaciones hoteleras.  
La API permite generar predicciones, validar los datos ingresados, mostrar errores claros y ejecutarse dentro de Docker.

También se agregó una interfaz tipo formulario para que un usuario no técnico pueda probar el modelo sin escribir JSON manualmente.

---

## 1. Cumplimiento de requisitos del Rol API Engineer

| Requisito del rol | Estado | Evidencia breve |
|---|---:|---|
| 1. Crear el servicio FastAPI con endpoints `/health`, `/predict` y `/version`. | ✅ Cumplido | Implementado en `api/main.py`. También se agregó `/` como formulario amigable. |
| 2. Definir los schemas de entrada/salida con Pydantic según `handoff/contracts/`. | ✅ Cumplido | Implementado en `api/schemas.py`. Se validan campos requeridos, tipos de datos, rangos y opciones permitidas. |
| 3. Cargar el modelo y el preproc pipeline `.pkl` una sola vez al iniciar el proceso. | ✅ Cumplido | Implementado en `api/inference.py`. Se carga `models/final_pipeline.pkl`, que contiene el preprocesamiento y el modelo. |
| 4. Implementar validación de inputs y manejo de errores con códigos HTTP claros. | ✅ Cumplido | La API devuelve `200` para predicciones correctas, `422` para errores de validación, `400` para reglas de negocio y `500` para errores internos. |
| 5. Escribir el Dockerfile con imagen `python:3.10-slim` y `requirements.txt` ajustado. | ✅ Cumplido | Se crearon `api/Dockerfile` y `api/requirements.txt`. |
| 6. Probar localmente con `docker build` + `docker run` y `curl` contra los contratos. | ✅ Cumplido | Se construyó la imagen `hotel-cancel-api:latest`, se ejecutó el contenedor y `curl http://localhost:8001/health` respondió `200 OK` con `{"status":"ok"}`. |

---

## 2. Cumplimiento de buenas prácticas recomendadas

| Buena práctica | Estado | Evidencia breve |
|---|---:|---|
| 1. Usar `python:3.10-slim`. | ✅ Cumplido | Definido en `api/Dockerfile`. |
| 2. Instalar primero `requirements.txt` y luego copiar código. | ✅ Cumplido | El Dockerfile copia primero `api/requirements.txt`, instala dependencias y luego copia el código. |
| 3. Correr como usuario no-root. | ✅ Cumplido | El Dockerfile crea y usa el usuario `appuser`. |
| 4. Exponer un único puerto (`EXPOSE 8000`). | ✅ Cumplido | El contenedor expone internamente solo el puerto `8000`. |
| 5. Definir `HEALTHCHECK`. | ✅ Cumplido | El Dockerfile consulta `/health` para validar si el contenedor está funcionando. |
| 6. No incluir modelos innecesarios si pesan mucho. | ✅ Cumplido para MVP | La imagen copia únicamente `models/final_pipeline.pkl`. Los demás modelos y carpetas pesadas se excluyen con `.dockerignore`. |

---

## 3. Recursos principales que deja la API

| Recurso | Ubicación / endpoint | Uso |
|---|---|---|
| Código principal de la API | `api/main.py` | Define endpoints, formulario web y manejo de errores. |
| Schemas de validación | `api/schemas.py` | Define qué datos acepta la API y qué respuesta devuelve. |
| Lógica de predicción | `api/inference.py` | Carga el pipeline y genera predicciones. |
| Dockerfile | `api/Dockerfile` | Permite construir la imagen Docker. |
| Dependencias | `api/requirements.txt` | Lista de paquetes necesarios para ejecutar la API. |
| Docker ignore | `.dockerignore` | Evita copiar archivos innecesarios al build Docker. |
| Modelo final usado | `models/final_pipeline.pkl` | Pipeline completo con preprocesamiento y modelo. |
| Interfaz amigable | `GET /` | Formulario para usuarios no técnicos. |
| Health check | `GET /health` | Verifica si la API está activa. |
| Metadata | `GET /version` | Muestra información de la API y del modelo. |
| Predicción | `POST /predict` | Endpoint principal para dashboard y otros sistemas. |
| Documentación técnica | `GET /docs` | Swagger UI para probar la API. |

---

## 4. Ejemplo de uso de `/predict`

### Request esperado

    {
      "hotel": "City Hotel",
      "lead_time": 120,
      "arrival_date": "2017-07-15",
      "stays_in_weekend_nights": 1,
      "stays_in_week_nights": 3,
      "adults": 2,
      "children": 0,
      "babies": 0,
      "meal": "BB",
      "market_segment": "Online TA",
      "distribution_channel": "TA/TO",
      "is_repeated_guest": 0,
      "previous_cancellations": 0,
      "previous_bookings_not_canceled": 0,
      "reserved_room_type": "A",
      "assigned_room_type": "A",
      "booking_changes": 0,
      "deposit_type": "No Deposit",
      "days_in_waiting_list": 0,
      "customer_type": "Transient",
      "adr": 95.5,
      "required_car_parking_spaces": 0,
      "total_of_special_requests": 1
    }

### Response esperado

    {
      "prediction": 0,
      "probability": 0.142902210354805,
      "risk_level": "bajo",
      "recommendation": "No se requiere acción urgente."
    }

| Campo | Significado |
|---|---|
| `prediction` | `0` = probablemente no cancela, `1` = posible cancelación. |
| `probability` | Probabilidad estimada de cancelación. |
| `risk_level` | Nivel de riesgo: `bajo`, `medio` o `alto`. |
| `recommendation` | Recomendación para el usuario. |

---

## 5. Comandos útiles para reproducir

### Ejecutar localmente sin Docker

    python -m uvicorn api.main:app --reload --host 0.0.0.0 --port 8000

Abrir en navegador:

    http://localhost:8000/

### Construir imagen Docker

    docker build -f api/Dockerfile -t hotel-cancel-api .

### Ejecutar contenedor Docker

    docker run --rm -p 8001:8000 hotel-cancel-api

Nota: dentro del contenedor la API corre en el puerto `8000`. El puerto `8001` se usó solo para prueba local.

### Probar health check

    curl http://localhost:8001/health

Resultado esperado:

    {"status":"ok"}

---

## 6. Guía breve para los roles siguientes

Esta sección solo indica qué recursos del Rol API Engineer podrían usar los siguientes roles.  
Cuando no hay un recurso directo de la API para un requisito, se coloca `-`.

---

## Rol 3: Cloud / DevOps Engineer

| Requisito del Rol 3 | Recurso de mi API que podría usar |
|---|---|
| 1. Decidir arquitectura: EC2 vs Lambda + API Gateway. | `api/Dockerfile`, `api/requirements.txt`, `models/final_pipeline.pkl`, `GET /health` |
| 2. Crear el repositorio ECR y subir la imagen Docker. | `api/Dockerfile`, `.dockerignore`, imagen `hotel-cancel-api:latest` |
| 3. Configurar IAM roles mínimos. | - |
| 4. Almacenar secretos en Secrets Manager o Parameter Store. | - |
| 5. Subir el modelo final a un bucket S3 versionado. | `models/final_pipeline.pkl` |
| 6. Crear pipeline CI/CD con GitHub Actions: push main → build → ECR → deploy. | `api/Dockerfile`, `api/requirements.txt`, `.dockerignore`, `GET /health` |
| 7. Configurar HTTPS y endpoint público estable. | `POST /predict`, `GET /health`, `GET /version`, `GET /` |

---

## Rol 4: Integration Engineer

| Requisito del Rol 4 | Recurso de mi API que podría usar |
|---|---|
| 1. Modificar `dashboard/app.py` para que consuma la API REST en lugar del modelo local. | `POST /predict` |
| 2. Configurar la URL del endpoint y la API key vía variables de entorno. | URL de `/predict`. Ejemplo local: `http://localhost:8001/predict` |
| 3. Implementar cache local con `st.cache_data`. | Response de `POST /predict` |
| 4. Manejar errores de red con mensajes amigables al usuario. | Errores de `/predict`: `422`, `400`, `500` |
| 5. Desplegar el dashboard en Streamlit Cloud, ECS Fargate o EC2 separado. | Endpoint público futuro de `POST /predict` |
| 6. Validar flujo end-to-end: usuario sube CSV → dashboard llama API → muestra predicciones + SHAP. | Contrato de entrada y salida de `POST /predict` |

---

## Pruebas mencionadas para Integration Engineer

| Prueba | Recurso de mi API que podría usar |
|---|---|
| Smoke test: `GET /health` responde 200 y latencia <1s. | `GET /health` |
| Contrato: `POST /predict` con example request produce formato esperado. | `POST /predict` |
| Carga básica: 100 requests concurrentes. | `POST /predict` |
| Timeout: si el modelo tarda >2s, el dashboard muestra mensaje claro. | `POST /predict` |
| Auth: peticiones sin API key son rechazadas. | - |
| Rollback: volver a la versión anterior si el deploy falla el smoke test. | `GET /health`, `GET /version`, imagen Docker |

---

## Rol 5: Site Reliability & Monitoring

| Requisito del Rol 5 | Recurso de mi API que podría usar |
|---|---|
| 1. Configurar logs estructurados JSON en la API y enviarlos a CloudWatch Logs. | Logs actuales de FastAPI/Uvicorn |
| 2. Definir métricas clave: latencia, throughput y tasa de error 5xx. | `POST /predict`, `GET /health` |
| 3. Crear dashboard CloudWatch con métricas + CPU/memoria del contenedor. | Contenedor Docker de la API |
| 4. Configurar alarmas: error rate >2% durante 5 min → SNS. | Códigos HTTP de la API: `200`, `400`, `422`, `500` |
| 5. Establecer monitoreo de drift: distribución de inputs vs baseline del Sprint 2. | Datos recibidos por `POST /predict` |
| 6. Documentar runbook: incidentes, rollback y escalamiento. | `GET /health`, `GET /version`, imagen Docker |

---

## 7. Sobre API Key

La API actual no implementa API Key. Esto fue intencional para mantener el alcance del Rol API Engineer enfocado en dejar funcionando la API, el modelo, las validaciones, Docker y las pruebas locales.

La API Key debería añadirse en una etapa posterior, cuando exista una arquitectura cloud y un endpoint público definido. En ese momento, el secreto debería manejarse con herramientas seguras como `AWS Secrets Manager` o `AWS Parameter Store`.

No se recomienda escribir claves directamente en el código, en el Dockerfile ni en el dashboard.

| Tema | Estado actual |
|---|---|
| API funcional | ✅ Sí |
| Docker funcionando | ✅ Sí |
| Validaciones | ✅ Sí |
| Mensajes de error claros | ✅ Sí |
| API Key | ⏳ Pendiente para Cloud / DevOps |

---

## 8. Resumen final

| Rol | Qué puede aprovechar de lo entregado |
|---|---|
| Cloud / DevOps | Dockerfile, requirements, `.dockerignore`, health check y modelo final. |
| Integration | Endpoint `/predict`, contrato de entrada/salida y errores claros. |
| Site Reliability / Monitoring | `/health`, `/version`, códigos HTTP y contenedor Docker. |

En resumen, el Rol API Engineer deja una API funcional, validada, dockerizada y lista para continuar con despliegue, integración y monitoreo.